<a href="https://colab.research.google.com/github/swrobuts/dav/blob/main/notebooks/07_Daten_Zusammenfuehren.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔗 Notebook 07: Daten zusammenführen

**Szenario**: Füge Kundendaten aus 3 verschiedenen Quellen zusammen.

**Lernziele:**
- pd.merge() mit allen Join-Typen
- SQL-Analogien verstehen
- pd.concat() für Zeilen und Spalten
- Daten in PostgreSQL laden und SQL-Joins üben

---

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Setup abgeschlossen!")


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Datensätze laden
customers = pd.read_csv(BASE_URL + "customers.csv")
orders = pd.read_csv(BASE_URL + "orders.csv")
products = pd.read_csv(BASE_URL + "products.csv")

print(f'customers: {customers.shape}')
print(f'orders: {orders.shape}')
print(f'products: {products.shape}')
print()
print('customers:', list(customers.columns))
print('orders:', list(orders.columns))
print('products:', list(products.columns))

## 1️⃣ pd.merge() – SQL-style Joins

In [ ]:
print('=== Merge-Typen im Vergleich ===')

# INNER JOIN (nur Übereinstimmungen)
df_inner = pd.merge(customers, orders, on='customer_id', how='inner')
print(f'Inner Join:  {len(df_inner):>5} Zeilen (nur Kunden MIT Bestellungen)')

# LEFT JOIN (alle Kunden, auch ohne Bestellungen)
df_left = pd.merge(customers, orders, on='customer_id', how='left')
print(f'Left Join:   {len(df_left):>5} Zeilen (alle Kunden)')
print(f'  Kunden ohne Bestellungen: {df_left["order_id"].isna().sum()}')

# RIGHT JOIN
df_right = pd.merge(customers, orders, on='customer_id', how='right')
print(f'Right Join:  {len(df_right):>5} Zeilen (alle Bestellungen)')

# OUTER JOIN
df_outer = pd.merge(customers, orders, on='customer_id', how='outer')
print(f'Outer Join:  {len(df_outer):>5} Zeilen (alles)')

In [ ]:
# SQL-Analogie
print('=== SQL-Analogien ===')
print()
print('-- INNER JOIN:')
print('SELECT * FROM customers c')
print('INNER JOIN orders o ON c.customer_id = o.customer_id')
print()
print('-- LEFT JOIN:')
print('SELECT * FROM customers c')
print('LEFT JOIN orders o ON c.customer_id = o.customer_id')
print()

# Dreifach-Join: Kunden → Bestellungen → Produkte
df_full = pd.merge(
    pd.merge(customers, orders, on='customer_id', how='left'),
    products, on='product_id', how='left'
)
print(f'Dreifach-Join: {df_full.shape}')
print(df_full[['vorname', 'nachname', 'city', 'bestelldatum', 'produktname', 'preis']].head(5))

## 2️⃣ pd.concat() – Zeilen und Spalten stapeln

In [ ]:
# Zeilenweise: Bestellungen aus verschiedenen Jahren
df_2023 = orders[orders['bestelldatum'] < '2023-07-01'].copy()
df_2024 = orders[orders['bestelldatum'] >= '2023-07-01'].copy()
df_2024['bestelldatum'] = df_2024['bestelldatum'].str.replace('2023','2024')

df_combined = pd.concat([df_2023, df_2024], axis=0, ignore_index=True)
print(f'df_2023: {len(df_2023)} | df_2024: {len(df_2024)} | concat: {len(df_combined)}')
print()

# Spaltenweise
df_basis = customers[['customer_id','vorname','nachname']].copy()
df_stats = customers.groupby('customer_id').size().reset_index(name='bestellungen')
df_col_combined = pd.concat([df_basis.set_index('customer_id'),
                              df_stats.set_index('customer_id')], axis=1).reset_index()
print('Spaltenweise concat:')
print(df_col_combined.head(3))

## 3️⃣ In SQLite laden & SQL-Joins üben

In [ ]:
import urllib.request
urllib.request.urlretrieve(BASE_URL + "dav_sample.db", "/tmp/dav_sample.db")
conn = sqlite3.connect('/tmp/dav_sample.db')

# Tabellen laden
customers.to_sql('customers', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)

# SQL-Join testen
query = """
    SELECT
        c.vorname || ' ' || c.nachname AS kunde,
        c.city,
        COUNT(o.order_id) AS anzahl_bestellungen,
        ROUND(SUM(p.preis * o.menge), 2) AS gesamtumsatz
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    LEFT JOIN products p ON o.product_id = p.product_id
    GROUP BY c.customer_id
    HAVING anzahl_bestellungen > 0
    ORDER BY gesamtumsatz DESC
    LIMIT 10
"""

df_sql = pd.read_sql(query, conn)
print('Top 10 Kunden nach Gesamtumsatz:')
print(df_sql)
conn.close()


## ✅ Challenges

1. Finde alle Kunden, die KEINE Bestellungen haben (LEFT JOIN + Nullwert-Filter)
2. Erstelle eine Tabelle: Umsatz nach Stadt und Kundenstatus
3. Postgres-Bonus: Lade alle 3 Tabellen in PostgreSQL und führe dort einen JOIN aus

In [ ]:
# Challenge 1:
# Kunden ohne Bestellungen (LEFT JOIN + NULL-Filter)
no_orders = pd.merge(customers, orders, on='customer_id', how='left')
no_orders = no_orders[no_orders['order_id'].isna()]
print(f'Kunden ohne Bestellungen: {len(no_orders)}')
print(no_orders[['customer_id','vorname','nachname']].head())

# Challenge 2:
# Umsatz nach Stadt
df_city = df_full.groupby('city').agg(
    umsatz=('preis', lambda x: (x * df_full.loc[x.index, 'menge']).sum())
).reset_index().sort_values('umsatz', ascending=False)
print('\nUmsatz nach Stadt:')
print(df_city.head(10))

# Challenge 3: SQLite statt PostgreSQL
conn2 = sqlite3.connect('/tmp/dav_sample.db')
query2 = """
    SELECT c.city, COUNT(DISTINCT c.customer_id) as kunden,
           ROUND(SUM(p.preis * o.menge), 2) as umsatz
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN products p ON o.product_id = p.product_id
    GROUP BY c.city ORDER BY umsatz DESC
"""
df_city_sql = pd.read_sql(query2, conn2)
conn2.close()
print('\nSQL-Ergebnis Umsatz nach Stadt:')
print(df_city_sql.head(10))


---
**Weiter mit:** `08_Visualisierung.ipynb`